# 🏋️ Data Collection & Web Scraping — Practical Exercises
> **Track:** Prime AI/ML Batch  
> **Topic:** Hands-on APIs, HTML Table Extraction, Hyperlinks & Scraper Pipelines  
> **Language / Notes Mode:** Easy Natural Hinglish  

---

Har question ke sath **Problem Statement**, **Hinglish Logic Explanation**, aur **Working Code Solution** included hai.


## ❓ Problem 1: Public REST API Consumption & JSON Normalization

### 📌 Problem Statement:
Ek public quotes API simulation ya live endpoint (`https://dummyjson.com/quotes?limit=5`) se quotes fetch karo.
1. `requests.get` se status code verify karo.
2. `pd.json_normalize` use karke quotes aur author ka ek clean DataFrame banao.

### 💡 Hinglish Explanation:
- Public APIs JSON return karti hain jisme metadata aur `quotes` array hota hai.
- `pd.json_normalize(json_data['quotes'])` seedha dictionary array ko rows me unpack kar deta hai.


In [ ]:
import requests
import pandas as pd

api_url = "https://dummyjson.com/quotes?limit=5"

try:
    res = requests.get(api_url, timeout=5)
    if res.status_code == 200:
        data = res.json()["quotes"]
        df_quotes = pd.json_normalize(data)[["id", "quote", "author"]]
        print("--- Quotes DataFrame from API ---\n", df_quotes)
    else:
        raise Exception("Status not 200")
except Exception:
    # Offline fallback
    mock_quotes = [
        {"id": 1, "quote": "Life isn't about finding yourself. Life is about creating yourself.", "author": "George Bernard Shaw"},
        {"id": 2, "quote": "Success is not final, failure is not fatal.", "author": "Winston Churchill"},
        {"id": 3, "quote": "Simplicity is the soul of efficiency.", "author": "Austin Freeman"}
    ]
    df_quotes = pd.DataFrame(mock_quotes)
    print("--- Quotes DataFrame (Local Fallback) ---\n", df_quotes)


## ❓ Problem 2: HTML Tabular Data Extraction into DataFrame

### 📌 Problem Statement:
Niche diye gaye raw HTML table string me se:
1. `BeautifulSoup` use karke table headers `<th>` aur table rows `<tr>/<td>` extract karo.
2. Inhe directly ek structured Pandas DataFrame me convert karo.

### 💡 Hinglish Explanation:
- Tables me pehla row aksar headers hota hai (`th`).
- Uske baad har row (`tr`) ke andar ke data cells (`td`) extract karke list me append karte hain.


In [ ]:
from bs4 import BeautifulSoup

html_table_snippet = """
<table class="market-data">
    <thead>
        <tr><th>Ticker</th><th>Company</th><th>Stock_Price</th><th>Change</th></tr>
    </thead>
    <tbody>
        <tr><td>NVDA</td><td>Nvidia Corp</td><td>142.50</td><td>+4.2%</td></tr>
        <tr><td>AAPL</td><td>Apple Inc</td><td>230.10</td><td>+1.1%</td></tr>
        <tr><td>MSFT</td><td>Microsoft Corp</td><td>428.80</td><td>-0.5%</td></tr>
        <tr><td>GOOGL</td><td>Alphabet Inc</td><td>168.40</td><td>+2.0%</td></tr>
    </tbody>
</table>
"""

soup = BeautifulSoup(html_table_snippet, "html.parser")

# 1. Extract Headers
headers = [th.get_text(strip=True) for th in soup.select("table.market-data th")]

# 2. Extract Data Rows
rows = []
for tr in soup.select("table.market-data tbody tr"):
    cells = [td.get_text(strip=True) for td in tr.find_all("td")]
    rows.append(cells)

# 3. Create DataFrame
df_stocks = pd.DataFrame(rows, columns=headers)
print("--- Extracted Stocks Market Table ---\n", df_stocks)


## ❓ Problem 3: Extracting Hyperlinks & Image URLs (Attributes Extraction)

### 📌 Problem Statement:
HTML webpage me se:
1. Saare `<a>` tags dhundho aur unke link text aur `href` URL nikal kar filter karo ki kaunse internal links hain aur kaunse external (`https://`).
2. Saari images `<img>` ke `src` attributes extract karo.

### 💡 Hinglish Explanation:
- Tag ke text ke liye `.get_text()` use hota hai, jabki attribute ke liye dictionary syntax `tag['href']` ya `tag.get('src')` use hota hai.


In [ ]:
nav_html = """
<div class="blog-post">
    <h2>Top AI Breakthroughs</h2>
    <img src="/static/images/banner.jpg" alt="AI Banner">
    <p>Read more on our <a href="/tutorials/ai-intro">Internal AI Tutorial</a>.</p>
    <p>Official docs at <a href="https://pytorch.org">PyTorch Official</a>.</p>
    <p>Follow updates on <a href="https://github.com">GitHub Repository</a>.</p>
    <img src="https://example.com/icons/logo.png" alt="Company Logo">
</div>
"""

soup_nav = BeautifulSoup(nav_html, "html.parser")

# 1. Hyperlinks
links_data = []
for a in soup_nav.find_all("a"):
    href = a.get("href", "")
    text = a.get_text(strip=True)
    is_external = href.startswith("http")
    links_data.append({"Text": text, "URL": href, "External": is_external})

print("Extracted Links Summary:")
display(pd.DataFrame(links_data))

# 2. Image Sources
image_srcs = [img.get("src") for img in soup_nav.find_all("img")]
print("\nExtracted Image Sources:\n", image_srcs)


## ❓ Problem 4: Rate-Limited Safe Scraper with `time.sleep`

### 📌 Problem Statement:
Multiple pages (e.g. Page 1, Page 2, Page 3) ko scrape karte waqt server par heavy load na pade iske liye:
1. `time.sleep(1)` delay implement karo.
2. Error handling `try-except` block lagao taaki ek page fail hone par poora script crash na ho.

### 💡 Hinglish Explanation:
- Professional web scraping ka rule hota hai "Polite Scraping". Har request ke baad thoda pause lagane se IP block hone ka risk kam ho jata hai.


In [ ]:
import time

pages = [1, 2, 3]
scraped_logs = []

for page in pages:
    page_url = f"https://example.com/products?page={page}"
    print(f"Scraping {page_url}...")
    
    # Polite delay
    time.sleep(0.5)
    
    # Simulation of scraping status
    scraped_logs.append({
        "Page": page,
        "Status": "Success (200)",
        "Items_Found": 10
    })

print("\n--- Scraping Execution Log ---")
display(pd.DataFrame(scraped_logs))


## ❓ Problem 5: E-Commerce Product Cards Scraper to Clean CSV

### 📌 Problem Statement:
Niche diye gaye 3 Product Cards HTML me se:
- Title
- Price (number me clean karke)
- Rating
- Stock Status

Inhe extract karo, clean types assign karo aur `products_catalog.csv` me save karo.

### 💡 Hinglish Explanation:
- Price me currency symbol (`$`) hata kar use float me convert karenge taaki analysis ho sake.


In [ ]:
catalog_html = """
<div class="products-grid">
    <div class="product-item">
        <h4 class="title">Wireless Noise-Cancelling Headphones</h4>
        <div class="price-box"><span class="currency">$</span><span class="amount">199.99</span></div>
        <div class="rating">4.8 / 5.0</div>
        <span class="stock in-stock">In Stock</span>
    </div>
    <div class="product-item">
        <h4 class="title">Mechanical Gaming Keyboard</h4>
        <div class="price-box"><span class="currency">$</span><span class="amount">89.50</span></div>
        <div class="rating">4.5 / 5.0</div>
        <span class="stock in-stock">In Stock</span>
    </div>
    <div class="product-item">
        <h4 class="title">Ultra-Wide 4K Monitor</h4>
        <div class="price-box"><span class="currency">$</span><span class="amount">549.00</span></div>
        <div class="rating">4.9 / 5.0</div>
        <span class="stock out-of-stock">Out of Stock</span>
    </div>
</div>
"""

soup_cat = BeautifulSoup(catalog_html, "html.parser")
products = []

for item in soup_cat.select("div.product-item"):
    title = item.select_one("h4.title").get_text(strip=True)
    price_val = float(item.select_one("span.amount").get_text(strip=True))
    rating_str = item.select_one("div.rating").get_text(strip=True)
    stock = item.select_one("span.stock").get_text(strip=True)
    
    products.append({
        "Product_Title": title,
        "Price_USD": price_val,
        "Rating": rating_str,
        "Stock_Status": stock
    })

df_catalog = pd.DataFrame(products)
print("--- Scraped Products Catalog ---")
display(df_catalog)

# Save to CSV
df_catalog.to_csv("products_catalog.csv", index=False)
print("\n✅ Saved to 'products_catalog.csv' successfully.")
